## PRÁCTICA 2: Aprendizaje Semi-supervisado y en Una Clase

### Cargar dataset

##### Trabajaremos con el conjunto de datos CIFAR-100 (50.000 instancias para entrenamiento y 10.000 para test), en el cual eliminaremos el 80% de sus etiquetas y realizaremos las siguientes particiones:

#####    40.000 instancias de entrenamiento no etiquetadas
#####    10.000 instancias de entrenamiento etiquetadas
#####    10.000 instancias de test etiquetadas

In [1]:
import tensorflow as tf
import numpy as np

In [2]:
labeled_data = 0.2 # Vamos a usar el etiquetado de sólo el 20% de los datos
np.random.seed(42)

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()

indexes = np.arange(len(x_train))
np.random.shuffle(indexes)
ntrain_data = int(labeled_data*len(x_train))
unlabeled_train = x_train[indexes[ntrain_data:]]
x_train = x_train[indexes[:ntrain_data]]
y_train = y_train[indexes[:ntrain_data]]

In [3]:
print(x_train.shape)
print(unlabeled_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(10000, 32, 32, 3)
(40000, 32, 32, 3)
(10000, 1)
(10000, 32, 32, 3)
(10000, 1)


#### Preprocesado de los datos

In [4]:
# convertimos de uint8 a float32 y normalizamos en rango [0, 1]
x_train = x_train.reshape(-1, 32, 32, 3).astype("float32") / 255.0
x_test = x_test.reshape(-1, 32, 32, 3).astype("float32") / 255.0


#### 1. Entrena un modelo, creado sobre TensorFlow, haciendo uso únicamente de las instancias etiquetadas de entrenamiento. Dicho modelo debe de tener al menos cuatro capas densas y/o convolucionales.

In [5]:
miCNN = tf.keras.models.Sequential([
            tf.keras.layers.Conv2D(32, (3, 3), activation="relu", input_shape=(32, 32, 3)),
            tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
            tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
            tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(100, activation="softmax")
        ])

optimizer = tf.keras.optimizers.Adam()

miCNN.compile(
            loss="sparse_categorical_crossentropy",
            optimizer=optimizer,
            metrics=["accuracy"]
        )

miCNN.fit(x_train, y_train, epochs=20, batch_size=32, verbose=1)


c:\Users\Usuario\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.0355 - loss: 4.3842
Epoch 2/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.0939 - loss: 3.9186
Epoch 3/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.1561 - loss: 3.5705
Epoch 4/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.2067 - loss: 3.2969
Epoch 5/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.2446 - loss: 3.0946
Epoch 6/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.2759 - loss: 2.9109
Epoch 7/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.3070 - loss: 2.7482
Epoch 8/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.3404 - loss: 2.5844
Epoch 9/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.3767 - loss: 2.4373
Epoch 10/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.4058 - loss: 2.2913
Epoch 11/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.4425 - loss: 2.1423
Epoch 12/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step

#### Responde las siguientes preguntas:
####    a. ¿Qué red has escogido? ¿Por qué? ¿Cómo la has entrenado?
####    b. ¿Cuál es el rendimiento del modelo en entrenamiento? ¿Y en prueba?
####    c. ¿Qué conclusiones sacas de los resultados detallados en el punto anterior?

#### 2. Entrena el mismo modelo, incorporando las instancias no etiquetadas de entrenamiento mediante la técnica de auto-aprendizaje. Opcionalmente, se ponderará cada instancia de entrada en función de su calidad (o certeza).

In [5]:
def miCNN_func():
    miCNN = tf.keras.models.Sequential([
                tf.keras.layers.Conv2D(32, (3, 3), activation="relu", input_shape=(32, 32, 3)),
                tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
                tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
                tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
                tf.keras.layers.Flatten(),
                tf.keras.layers.Dense(128, activation="relu"),
                tf.keras.layers.Dense(100, activation="softmax")
            ])

    optimizer = tf.keras.optimizers.Adam()

    miCNN.compile(
                loss="sparse_categorical_crossentropy",
                optimizer=optimizer,
                metrics=["accuracy"]
            )
    
    return miCNN

In [ ]:
def self_training(model_func, x_train, y_train, unlabeled_data, x_test, y_test, thresh=0.5, train_epochs=3):
    train_data = x_train.copy()
    train_label = y_train.copy()

    for i in range(train_epochs):
        model = model_func()
        print("Train Epoch: ", (i+1))
        model.fit(train_data, train_label, epochs=5,
                  batch_size=128, verbose=1)

        y_pred = model.predict(unlabeled_data)

        y_class = np.argmax(y_pred, axis=1)
        y_value = np.max(y_pred, axis=1)

        train_data = x_train.copy()
        train_label = y_train.copy()

        for x_u, y_c, y_v in zip(unlabeled_data, y_class, y_value):
            if y_v > thresh:
                x_u = x_u[np.newaxis, ...]   # (32,32,3) → (1,32,32,3)
                train_data = np.concatenate((train_data, x_u), axis=0)
                train_label = np.append(train_label, y_c)

    return model

In [7]:
model = self_training(
    miCNN_func,
    x_train[::10],
    y_train[::10],
    unlabeled_train[::10],
    x_test,
    y_test,
    thresh=0.9,
    train_epochs=3
)

c:\Users\Usuario\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Train Epoch:  0
Epoch 1/5
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.0150 - loss: 4.6153
Epoch 2/5
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.0250 - loss: 4.5703
Epoch 3/5
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.0370 - loss: 4.5275
Epoch 4/5
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.0500 - loss: 4.4616
Epoch 5/5
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.0490 - loss: 4.3703
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Train Epoch:  1
Epoch 1/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - accuracy: 0.3665 - loss: 10.1537
Epoch 2/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.5605 - loss: 1.7632
Epoch 3/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.6156 - loss: 1.5187
Epoch 4/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.6559 - loss: 1.3541
Epoch 5/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.6833 - loss: 1.2733
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Train Epoch:  2
Epoch 1/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s

#### Responde las siguientes preguntas:
####    a. ¿Qué parámetros has definido para el entrenamiento?
####    b. ¿Cuál es el rendimiento del modelo en entrenamiento? ¿Y en prueba?
####    c. ¿Se mejoran los resultados obtenidos en el Ejercicio 1?
####    d. ¿Qué conclusiones sacas de los resultados detallados en los puntos anteriores?

#### 3. Entrena un modelo de aprendizaje semisupervisado de tipo autoencoder en dos pasos (primero el autoencoder, después el clasificador). La arquitectura del encoder debe ser exactamente la misma que la definida en los Ejercicios 1 y 2, a excepción del último bloque de capas.

In [8]:
class MiAutoencoder:

    def __init__(self, input_shape):
        # encoder
        input_layer = tf.keras.layers.Input(shape=input_shape)
        x = tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same")(input_layer)
        x = tf.keras.layers.MaxPooling2D((2, 2), padding="same")(x)
        x = tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
        code = tf.keras.layers.MaxPooling2D((2, 2), padding="same")(x)
        # decoder
        x = tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same")(code)
        x = tf.keras.layers.UpSampling2D((2, 2))(x)
        x = tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same")(x)
        x = tf.keras.layers.UpSampling2D((2, 2))(x)
        output_layer = tf.keras.layers.Conv2D(3, (3, 3), activation="sigmoid", padding="same")(x)

        self.autoencoder = tf.keras.Model(input_layer, output_layer)
        self.encoder = tf.keras.Model(input_layer, code)
        self.decoder = tf.keras.Model(code, output_layer)

        self.autoencoder.compile(optimizer="adam", loss="mse")

    def fit(self, X, y=None, sample_weight=None):
        self.autoencoder.fit(
            X, X,
            epochs=10,
            batch_size=128,
            shuffle=True,
            sample_weight=sample_weight,
            verbose=1
        )

    def get_encoded_data(self, X):
        return self.encoder.predict(X)

    def __del__(self):
        tf.keras.backend.clear_session()

In [ ]:
class MiClasificador:

    def __init__(self, input_dim):
        self.model = tf.keras.Sequential([
            tf.keras.layers.Flatten(input_shape=(8, 8, 64)),    
            tf.keras.layers.Dense(128, activation="relu", input_shape=(input_dim,)),
            tf.keras.layers.Dense(100, activation="softmax")
        ])

        self.model.compile(
            optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    def fit(self, X, y, sample_weight=None):
        self.model.fit(
            X, y,
            epochs=10,
            batch_size=128,
            shuffle=True,
            sample_weight=sample_weight,
            verbose=1
        )

    def predict(self, X):
        probs = self.model.predict(X)
        return np.argmax(probs, axis=1)

    def predict_proba(self, X):
        return self.model.predict(X)

    def score(self, X, y):
        loss, acc = self.model.evaluate(X, y, verbose=0)
        return acc

    def __del__(self):
        tf.keras.backend.clear_session()

In [20]:
def semisupervised_training(autoencoder, classifier, x_train, y_train, unlabeled_data):

    all_data = np.concatenate((x_train, unlabeled_data), axis=0)
    autoencoder.fit(all_data)
    code = autoencoder.get_encoded_data(x_train)
    classifier.fit(code, y_train)

    return autoencoder, classifier

In [21]:
autoencoder = MiAutoencoder(input_shape=x_train.shape[1:])
classifier = MiClasificador(input_dim=64)  # tamaño del code

autoencoder, classifier = semisupervised_training(
    autoencoder,
    classifier,
    x_train,
    y_train,
    unlabeled_train
)

pred_data = autoencoder.get_encoded_data(x_test)
print("Test accuracy :", classifier.score(pred_data, y_test))

Epoch 1/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 19s 45ms/step - loss: 15438.4355
Epoch 2/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 18s 45ms/step - loss: 15438.0352
Epoch 3/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 18s 45ms/step - loss: 15438.0361
Epoch 4/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 17s 45ms/step - loss: 15438.0381
Epoch 5/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 20s 51ms/step - loss: 15438.0381
Epoch 6/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 19s 48ms/step - loss: 15438.0439
Epoch 7/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 18s 45ms/step - loss: 15438.0381
Epoch 8/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 18s 46ms/step - loss: 15438.0361
Epoch 9/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 18s 45ms/step - loss: 15438.0264
Epoch 10/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 18s 45ms/step - loss: 15438.0352
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Epoch 1/4
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0284 - loss: 4.5182
Epoch 2/4
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0646 - loss: 4.2428
Epoch 3/4
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0

#### Responde las siguientes preguntas:
####    a. ¿Cuál es la arquitectura del modelo? ¿Y sus hiperparámetros?
####    b. ¿Cuál es el rendimiento del modelo en entrenamiento? ¿Y en prueba?
####    c. ¿Se mejoran los resultados obtenidos en los Ejercicios 1 y 2?
####    d. ¿Qué conclusiones sacas de los resultados detallados en los puntos anteriores?

#### 4. Entrena un modelo de aprendizaje semisupervisado de tipo autoencoder en un paso (autoencoder y clasificador al mismo tiempo). La arquitectura del autoencoder será la misma que la definida en el Ejercicio 3, y la combinación de encoder y clasificador será igual a la arquitectura definida en el Ejercicio 1.

In [12]:
class MiClasificadorSemisupervisado:
    def __init__(self, input_shape):
        inputs = tf.keras.layers.Input(shape=input_shape)
        # encoder
        x = tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same")(inputs)
        x = tf.keras.layers.MaxPooling2D((2, 2), padding="same")(x)
        x = tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
        code = tf.keras.layers.MaxPooling2D((2, 2), padding="same")(x)
        # decoder
        x_decoder = tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same")(code)
        x_decoder = tf.keras.layers.UpSampling2D((2, 2))(x_decoder)
        x_decoder = tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same")(x_decoder)
        x_decoder = tf.keras.layers.UpSampling2D((2, 2))(x_decoder)
        decoding = tf.keras.layers.Conv2D(3, (3, 3), activation="sigmoid", padding="same", name="decoding")(x_decoder)
        # clasificador
        x_classifier = tf.keras.layers.Flatten()(code)
        x_classifier = tf.keras.layers.Dense(128, activation="relu")(x_classifier)
        classification = tf.keras.layers.Dense(100, activation="softmax", name="classification")(x_classifier)

        self.model = tf.keras.Model(inputs=inputs, outputs=[decoding, classification])

        self.model.compile(
            optimizer="adam",
            loss={"decoding": "mse", "classification": "sparse_categorical_crossentropy"},
            metrics={"classification": "accuracy"}
        )

    def fit(self, X, y, unlabeled_data):
        X_all = np.concatenate((X, unlabeled_data), axis=0)

        y_decoding = X_all    # la etiqueta es la propia entrada
        y_classification = np.concatenate((y.reshape(-1), np.zeros(len(unlabeled_data))))

        w_classification = np.concatenate((np.ones(len(X)), np.zeros(len(unlabeled_data))))

        self.model.fit(
            X_all,
            [y_decoding, y_classification],
            sample_weight=[None, w_classification],
            epochs=20,
            batch_size=256,
            shuffle=True,
            verbose=1
        )

    def predict(self, X):
        _, probs = self.model.predict(X)
        return np.argmax(probs, axis=1)

    def predict_proba(self, X):
        _, probs = self.model.predict(X)
        return probs

    def score(self, X, y):
        preds = self.predict(X)
        return np.mean(preds == y)

    def __del__(self):
        tf.keras.backend.clear_session()

In [13]:
def semisupervised_training_v2(model, x_train, y_train, unlabeled_data):
    model.fit(x_train, y_train, unlabeled_data)
    return model

In [14]:
model = MiClasificadorSemisupervisado(input_shape=x_train.shape[1:])

model = semisupervised_training_v2(
    model,
    x_train,
    y_train,
    unlabeled_train
)

print(model.score(x_test, y_test))

Epoch 1/20
196/196 ━━━━━━━━━━━━━━━━━━━━ 21s 101ms/step - classification_accuracy: 0.0031 - classification_loss: 0.9190 - decoding_loss: 15437.8467 - loss: 15441.3311
Epoch 2/20
196/196 ━━━━━━━━━━━━━━━━━━━━ 20s 103ms/step - classification_accuracy: 0.0123 - classification_loss: 0.8786 - decoding_loss: 15441.0195 - loss: 15438.9170
Epoch 3/20
196/196 ━━━━━━━━━━━━━━━━━━━━ 19s 98ms/step - classification_accuracy: 0.0317 - classification_loss: 0.8128 - decoding_loss: 15432.6211 - loss: 15438.8496
Epoch 4/20
196/196 ━━━━━━━━━━━━━━━━━━━━ 19s 97ms/step - classification_accuracy: 0.0423 - classification_loss: 0.7774 - decoding_loss: 15435.6699 - loss: 15438.8145
Epoch 5/20
196/196 ━━━━━━━━━━━━━━━━━━━━ 22s 111ms/step - classification_accuracy: 0.0493 - classification_loss: 0.7526 - decoding_loss: 15439.7529 - loss: 15438.7891
Epoch 6/20
196/196 ━━━━━━━━━━━━━━━━━━━━ 22s 113ms/step - classification_accuracy: 0.0521 - classification_loss: 0.7318 - decoding_loss: 15446.8545 - loss: 15438.7734
Epoch 

#### Responde las siguientes preguntas:
####    a. ¿Cuál es la arquitectura del modelo? ¿Y sus hiperparámetros?
####    b. ¿Cuál es el rendimiento del modelo en entrenamiento? ¿Y en prueba?
####    c. ¿Se mejoran los resultados obtenidos en los ejercicios anteriores?
####    d. ¿Qué conclusiones sacas de los resultados detallados en los puntos anteriores?

#### 5. Repite el mismo entrenamiento de los Ejercicios 2-4, pero eliminando las instancias no etiquetadas más atípicas con respecto a los datos etiquetados. Se cumplirán los siguientes puntos:

#### · La arquitectura de la red de clasificación en una clase será la misma a la utilizada en el clasificador del Ejercicio 1, a excepción de la capa de salida.

#### ·  Utiliza la técnica explicada en el Notebook 5, usando un valor de 𝑣 = 0,9.

#### Responde las siguientes preguntas:
####    a. ¿Se mejoran los resultados con respecto a los anteriores ejercicios? ¿Qué conclusiones sacas de estos resultados?

#### 6. Repite los Ejercicios 3-5 cambiando el autencoder por la técnica definida en el apartado “Hay vida más allá del autoencoder” del Notebook 4. Contesta a las preguntas de dichos ejercicios. Se cumplirán los siguientes puntos:

In [37]:
data_augmentation = tf.keras.models.Sequential(
    [
        tf.keras.layers.Input(shape=(32, 32, 3)),
        # tf.keras.layers.RandomFlip("horizontal"),  # Puede ser util en otros casos
        tf.keras.layers.RandomRotation(0.05),
        tf.keras.layers.RandomTranslation(0.15, 0.15),
        tf.keras.layers.RandomZoom(.15),
    ]
)

data_augmentation_2 = tf.keras.models.Sequential(
    [
        tf.keras.layers.Input(shape=(32, 32, 3)),
        # tf.keras.layers.RandomFlip("horizontal"),  # Puede ser util en otros casos
        tf.keras.layers.RandomTranslation(0.15, 0.15),
        tf.keras.layers.Resizing(48, 48), # para CIFAR, para MNIST usar 40 en lugar de 48
        tf.keras.layers.RandomCrop(32, 32), # para CIFAR, para MNIST usar 28 en lugar de 32
    ]
)

In [40]:
class ModeloAug:

    def __init__(self, tau=5.0, lamda=0.5):
        self.tau = tau
        self.lamda = lamda
        self.aug = data_augmentation
        self.aug2 = data_augmentation_2

        inputs = tf.keras.layers.Input(shape=(32, 32, 3))
        x = tf.keras.layers.Conv2D(32,3,activation='relu')(inputs)
        x = tf.keras.layers.MaxPooling2D()(x)
        x = tf.keras.layers.Conv2D(64,3,activation='relu')(x)
        x = tf.keras.layers.MaxPooling2D()(x)
        x = tf.keras.layers.Flatten()(x)
        code = tf.keras.layers.Dense(128, activation='relu')(x)
        cluster = tf.keras.layers.Dense(100, activation='softmax')(code)

        self.encoder = tf.keras.Model(inputs, code)
        self.cluster_model = tf.keras.Model(inputs, cluster)
        self.optimizer = tf.keras.optimizers.Adam()

    def loss_C(self, clustered1, clustered2):
        return tf.reduce_mean(clustered1*(1-clustered1) + clustered2*(1-clustered2))   # reduce_mean pasa de matriz a escalar

    def loss_M(self, encoded1, encoded2):
        M = tf.matmul(encoded1, encoded2, transpose_b=True)
        identity_matrix = tf.eye((tf.shape(encoded1)[0]))  # matriz identidad
        return tf.reduce_mean(
            tf.keras.losses.categorical_crossentropy(identity_matrix, (tf.nn.softmax(M/self.tau, axis=1)))
        )

    def fit(self, X, epochs=10, batch_size=128):
        dataset = tf.data.Dataset.from_tensor_slices(X).batch(batch_size)
        for epoch in range(epochs):
            for batch in dataset:
                # batch = tf.reshape(batch, (-1, 32, 32, 3))
                with tf.GradientTape() as tape:
                    augX1 = self.aug(batch)
                    augX2 = self.aug2(batch)

                    encoded1 = self.encoder(augX1)
                    encoded2 = self.encoder(augX2)
                    
                    clustered1 = self.cluster_model(augX1)
                    clustered2 = self.cluster_model(augX2)

                    lossC = self.loss_C(clustered1, clustered2)
                    lossM = self.loss_M(encoded1, encoded2)

                    loss = lossM + self.lamda*lossC

                vars = self.cluster_model.trainable_variables    # cogemos solo los pesos del modelo completo (encoder+cluster)
                grads = tape.gradient(loss, vars)     # calculamos los gradientes del loss respecto a cada peso
                self.optimizer.apply_gradients(zip(grads, vars))   # actualizamos pesos
            print("epoch:", epoch, "loss:", float(loss))

    def get_representation(self, X):
        return self.encoder.predict(X)

    def predict_clusters(self, X):
        probs = self.cluster_model.predict(X.reshape(-1, 32, 32, 3))
        return np.argmax(probs, axis=1)

    def __del__(self):
        tf.keras.backend.clear_session()

In [41]:
model = ModeloAug()
model.fit(x_train, epochs=10, batch_size=128)
clusters = model.predict_clusters(x_test)

epoch: 0 loss: 2.1809298992156982
epoch: 1 loss: 1.3286826610565186
epoch: 2 loss: 0.7210389375686646
epoch: 3 loss: 0.6401172280311584
epoch: 4 loss: 0.4468613564968109
epoch: 5 loss: 0.27840930223464966
epoch: 6 loss: 0.4101434648036957
epoch: 7 loss: 0.8175603151321411
epoch: 8 loss: 0.6112183332443237
epoch: 9 loss: 0.5525466203689575
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


#### Responde las siguientes preguntas:
####    a. La arquitectura de la red será igual a la parte encoder del autencoder definido en los ejercicios anteriores
####    b. El modelo debe entrenar correctamente.